# Clinical Imputation Pipeline

**Source script:** `pre-processing.py`

Notebook mirror for submission traceability.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor




DATA_PATH = "analysis_dataset.csv"
DATA_PATH_XLSX = "analysis_dataset.xlsx"
SHEET_NAME = None

OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DIAG_COL = "diagnosis"
DIAG_COL_CANDIDATES = ["group", "(suspected) diagnosis", "diagnosis"]

ID_COLS = ["ID"]
TARGET_COLS = []
DATE_COLS = ["date of presentation", "presentation_date", "date of birth"]

IGNORE_WEATHER_IN_IMPUTATION = True
WEATHER_PREFIXES = [
    "temperature_2m",
    "apparent_temperature",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation",
    "rain_sum",
    "snowfall_sum",
    "shortwave_radiation",
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m",
    "et0_fao_evapotranspiration",
    "uv_index_max",
    "surface_pressure_max",
]

HIGH_MISSING_THRESH = 0.40
JITTER_FRAC = 0.05
CLIP_MODE = "p01_p99"

RUN_BENCH = True
MASK_FRAC = 0.10
N_REPEATS = 5
MASKING = "stratified_by_diagnosis"
BENCHMARK_RANDOM_SEED = 42

BENCHMARK_METHODS = [
    "SIMPLE_MEDIAN",
    "STRATIFIED_DIAG_MEDIAN",
    "KNN_NUMERIC",
    "MICE_NUMERIC",
    "MISSFOREST_NUMERIC",
]

CATEGORICAL_MISSING_INDICATORS = False

CORE_LABS = {
    "initial hematocrit",
    "leukocyte count",
    "neutrophil count (x 10^9/L)",
    "lymphocyte count",
    "Initial total protein concentration (in g/l)",
    "Albumin concentration (in g/l)",
    "CREA concentration ",
    "BUN concentration",
    "K value",
    "Na value",
    "Cl value",
}

CORE_LAB_KEYWORDS = [
    "hematocrit",
    "leukocyte",
    "neutrophil",
    "lymphocyte",
    "protein",
    "albumin",
    "crea",
    "bun",
    "k value",
    "na value",
    "cl value",
]





def load_data():
    if Path(DATA_PATH).exists():
        return pd.read_csv(DATA_PATH)
    if Path(DATA_PATH_XLSX).exists():
        return pd.read_excel(DATA_PATH_XLSX, sheet_name=SHEET_NAME)
    raise FileNotFoundError("No input dataset found. Check DATA_PATH/DATA_PATH_XLSX.")


def choose_diag_col(columns):
    if DIAG_COL in columns:
        return DIAG_COL
    for col in DIAG_COL_CANDIDATES:
        if col in columns:
            return col
    return None


def is_weather_column(col):
    lower = col.lower()
    for prefix in WEATHER_PREFIXES:
        if lower.startswith(prefix) or prefix in lower:
            return True
    return False


def is_core_lab(col):
    if col in CORE_LABS:
        return True
    lower = col.lower()
    return any(keyword in lower for keyword in CORE_LAB_KEYWORDS)


def detect_numeric_columns(df, exclude_cols):
    numeric_cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        non_missing = df[col].notna().sum()
        converted_non_missing = converted.notna().sum()
        if non_missing > 0 and converted_non_missing / non_missing >= 0.9:
            df[col] = converted
            numeric_cols.append(col)
    return numeric_cols


def get_categorical_columns(df, exclude_cols, numeric_cols):
    categorical_cols = []
    for col in df.columns:
        if col in exclude_cols or col in numeric_cols:
            continue
        if df[col].dtype == "bool" or df[col].dtype.name == "category":
            categorical_cols.append(col)
        elif df[col].dtype == "object":
            categorical_cols.append(col)
    return categorical_cols


def compute_missingness(df):
    missing_n = df.isna().sum()
    pct = (missing_n / len(df)) * 100
    return pd.DataFrame({
        "variable": missing_n.index,
        "missing_n": missing_n.values,
        "missing_pct": pct.values,
    }).sort_values("missing_pct", ascending=False)


def mask_numeric_values(df, numeric_cols, diag_col, mask_frac, rng, stratified=True):
    masked_df = df.copy()
    mask_map = {col: np.zeros(len(df), dtype=bool) for col in numeric_cols}
    for col in numeric_cols:
        observed_idx = df.index[df[col].notna()]
        if observed_idx.empty:
            continue
        if stratified and diag_col and diag_col in df.columns:
            group_series = df.loc[observed_idx, diag_col]
            for group_val in group_series.unique():
                if pd.isna(group_val):
                    group_mask = group_series.isna()
                else:
                    group_mask = group_series == group_val
                group_idx = observed_idx[group_mask]
                if group_idx.empty:
                    continue
                n_mask = max(1, int(round(len(group_idx) * mask_frac)))
                n_mask = min(n_mask, len(group_idx))
                chosen = rng.choice(group_idx, size=n_mask, replace=False)
                mask_map[col][df.index.get_indexer(chosen)] = True
                masked_df.loc[chosen, col] = np.nan
        else:
            n_mask = max(1, int(round(len(observed_idx) * mask_frac)))
            n_mask = min(n_mask, len(observed_idx))
            chosen = rng.choice(observed_idx, size=n_mask, replace=False)
            mask_map[col][df.index.get_indexer(chosen)] = True
            masked_df.loc[chosen, col] = np.nan
    return masked_df, mask_map


def evaluate_imputation(original_df, imputed_df, mask_map, numeric_cols):
    rows = []
    for col in numeric_cols:
        mask = mask_map.get(col)
        if mask is None or not mask.any():
            continue
        true_vals = original_df[col].to_numpy()[mask]
        pred_vals = imputed_df[col].to_numpy()[mask]
        diff = pred_vals - true_vals
        rows.append({
            "variable": col,
            "mae": np.mean(np.abs(diff)),
            "rmse": np.sqrt(np.mean(diff ** 2)),
        })
    return pd.DataFrame(rows)


def impute_simple_median(df, numeric_cols):
    out = df.copy()
    for col in numeric_cols:
        if out[col].dropna().empty:
            continue
        out[col] = out[col].fillna(out[col].median(skipna=True))
    return out


def impute_stratified_median(df, numeric_cols, diag_col):
    out = df.copy()
    for col in numeric_cols:
        if out[col].dropna().empty:
            continue
        if diag_col and diag_col in out.columns:
            group_median = out.groupby(diag_col)[col].transform(
                lambda s: s.median(skipna=True) if s.notna().any() else np.nan
            )
            overall_median = out[col].median(skipna=True)
            out[col] = out[col].fillna(group_median).fillna(overall_median)
        else:
            out[col] = out[col].fillna(out[col].median(skipna=True))
    return out


def impute_knn_numeric(df, numeric_cols):
    out = df.copy()
    valid = [col for col in numeric_cols if out[col].notna().any()]
    if not valid:
        return out
    scaler = StandardScaler()
    scaled = scaler.fit_transform(out[valid])
    knn = KNNImputer(n_neighbors=5)
    imputed_scaled = knn.fit_transform(scaled)
    out[valid] = scaler.inverse_transform(imputed_scaled)
    return out


def impute_mice_numeric(df, numeric_cols):
    out = df.copy()
    valid = [col for col in numeric_cols if out[col].notna().any()]
    if not valid:
        return out
    imputer = IterativeImputer(
        max_iter=10,
        sample_posterior=True,
        random_state=42,
    )
    out[valid] = imputer.fit_transform(out[valid])
    return out


def impute_missforest_numeric(df, numeric_cols):
    out = df.copy()
    valid = [col for col in numeric_cols if out[col].notna().any()]
    if not valid:
        return out
    estimator = ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    )
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,
        sample_posterior=False,
        random_state=42,
    )
    out[valid] = imputer.fit_transform(out[valid])
    return out


def benchmark_numeric_methods(df, numeric_cols, diag_col):
    records = []
    stratified = MASKING == "stratified_by_diagnosis"
    for repeat in range(N_REPEATS):
        rng = np.random.default_rng(BENCHMARK_RANDOM_SEED + repeat)
        masked_df, mask_map = mask_numeric_values(
            df, numeric_cols, diag_col, MASK_FRAC, rng, stratified=stratified
        )
        for method in BENCHMARK_METHODS:
            if method == "SIMPLE_MEDIAN":
                imputed = impute_simple_median(masked_df, numeric_cols)
            elif method == "STRATIFIED_DIAG_MEDIAN":
                imputed = impute_stratified_median(masked_df, numeric_cols, diag_col)
            elif method == "KNN_NUMERIC":
                imputed = impute_knn_numeric(masked_df, numeric_cols)
            elif method == "MICE_NUMERIC":
                imputed = impute_mice_numeric(masked_df, numeric_cols)
            elif method == "MISSFOREST_NUMERIC":
                imputed = impute_missforest_numeric(masked_df, numeric_cols)
            else:
                continue
            scores = evaluate_imputation(df, imputed, mask_map, numeric_cols)
            scores["method"] = method
            scores["repeat"] = repeat + 1
            records.append(scores)
    if not records:
        return pd.DataFrame()
    return pd.concat(records, ignore_index=True)


def summarize_benchmark(scores_df):
    if scores_df.empty:
        return pd.DataFrame()
    agg = (
        scores_df
        .groupby(["variable", "method"])
        .agg(mean_mae=("mae", "mean"), mean_rmse=("rmse", "mean"))
        .reset_index()
    )
    return agg


def get_bench_best(agg_df):
    best_map = {}
    if agg_df.empty:
        return best_map
    for var, subset in agg_df.groupby("variable"):
        best = subset.sort_values(["mean_mae", "mean_rmse"]).iloc[0]
        best_map[var] = {
            "bench_best_method": best["method"],
            "bench_best_mae": best["mean_mae"],
            "bench_best_rmse": best["mean_rmse"],
        }
    return best_map


def add_categorical_missing(df, categorical_cols):
    out = df.copy()
    for col in categorical_cols:
        if out[col].isna().any():
            out[col] = out[col].fillna("Missing")
            if CATEGORICAL_MISSING_INDICATORS:
                out[f"{col}__missing"] = df[col].isna().astype(int)
    return out


def add_numeric_missing_flags(df, numeric_cols, original_df):
    out = df.copy()
    for col in numeric_cols:
        if original_df[col].isna().any():
            out[f"{col}__missing"] = original_df[col].isna().astype(int)
    return out


def apply_jitter_and_clip(imputed_df, original_df, col, rng):
    missing_mask = original_df[col].isna()
    if not missing_mask.any():
        return
    observed = original_df[col].dropna()
    if observed.empty:
        return
    median = observed.median()
    residuals = observed - median
    sd = residuals.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return
    jitter_scale = JITTER_FRAC * sd
    noise = rng.normal(0, jitter_scale, size=missing_mask.sum())
    imputed_df.loc[missing_mask, col] = imputed_df.loc[missing_mask, col] + noise

    if CLIP_MODE == "p01_p99":
        low = observed.quantile(0.01)
        high = observed.quantile(0.99)
    else:
        low = observed.min()
        high = observed.max()
    imputed_df.loc[missing_mask, col] = imputed_df.loc[missing_mask, col].clip(low, high)


def make_safe_filename(value):
    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", str(value))
    return safe.strip("_")


def plot_core_lab_histograms(original_df, final_df, core_labs):
    for col in core_labs:
        if col not in original_df.columns:
            continue
        orig = pd.to_numeric(original_df[col], errors="coerce")
        final = pd.to_numeric(final_df[col], errors="coerce")
        missing_mask = original_df[col].isna()
        imputed_only = final[missing_mask]

        if orig.dropna().empty:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
        sns.histplot(orig.dropna(), bins=30, color="#4C72B0", ax=axes[0])
        axes[0].set_title("Original (observed)")

        if imputed_only.dropna().empty:
            axes[1].text(0.5, 0.5, "No imputed values", ha="center", va="center")
            axes[1].set_axis_off()
        else:
            sns.histplot(imputed_only.dropna(), bins=30, color="#55A868", ax=axes[1])
            axes[1].set_title("Imputed (missing-only)")

        fig.suptitle(f"{col}")
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f"hist_{make_safe_filename(col)}.png", dpi=300)
        plt.close(fig)






def main():
    df = load_data()

    diag_col = choose_diag_col(df.columns)

    exclude_cols = set(ID_COLS + TARGET_COLS + DATE_COLS)

    numeric_cols = detect_numeric_columns(df, exclude_cols)
    categorical_cols = get_categorical_columns(df, exclude_cols, numeric_cols)

    if IGNORE_WEATHER_IN_IMPUTATION:
        numeric_cols = [c for c in numeric_cols if not is_weather_column(c)]
        categorical_cols = [c for c in categorical_cols if not is_weather_column(c)]

    missingness = compute_missingness(df)
    missingness.to_csv(OUTPUT_DIR / "missingness_table.csv", index=False)


    bench_best = {}
    bench_summary = pd.DataFrame()
    if RUN_BENCH and numeric_cols:
        bench_raw = benchmark_numeric_methods(df, numeric_cols, diag_col)
        bench_raw.to_csv(OUTPUT_DIR / "imputation_benchmark_raw.csv", index=False)
        bench_summary = summarize_benchmark(bench_raw)
        bench_summary.to_csv(OUTPUT_DIR / "imputation_benchmark_summary.csv", index=False)
        bench_best = get_bench_best(bench_summary)


    policy_rows = []
    missing_map = dict(zip(missingness["variable"], missingness["missing_pct"]))
    for col in numeric_cols:
        miss_pct = missing_map.get(col, 0.0) / 100.0
        core = is_core_lab(col)
        if miss_pct > HIGH_MISSING_THRESH or not core:
            chosen_method = "STRATIFIED_DIAG_MEDIAN" if diag_col else "SIMPLE_MEDIAN"
            chosen_reason = "high_missing_or_non_core"
        else:
            if col in bench_best:
                chosen_method = bench_best[col]["bench_best_method"]
                chosen_reason = "core_lab_bench_best"
            else:
                chosen_method = "MISSFOREST_NUMERIC"
                chosen_reason = "core_lab_default"
        bench_info = bench_best.get(col, {})
        policy_rows.append({
            "variable": col,
            "variable_type": "numeric",
            "missing_pct": missing_map.get(col, 0.0),
            "is_core_lab": core,
            "chosen_method": chosen_method,
            "chosen_reason": chosen_reason,
            "bench_best_method": bench_info.get("bench_best_method"),
            "bench_best_mae": bench_info.get("bench_best_mae"),
            "bench_best_rmse": bench_info.get("bench_best_rmse"),
        })

    for col in categorical_cols:
        policy_rows.append({
            "variable": col,
            "variable_type": "categorical",
            "missing_pct": missing_map.get(col, 0.0),
            "is_core_lab": False,
            "chosen_method": "CATEGORICAL_MISSING_LEVEL",
            "chosen_reason": "categorical_missing_level",
            "bench_best_method": None,
            "bench_best_mae": None,
            "bench_best_rmse": None,
        })

    policy_df = pd.DataFrame(policy_rows)
    policy_df.to_csv(OUTPUT_DIR / "imputation_policy.csv", index=False)


    imputed_simple = impute_simple_median(df, numeric_cols)
    imputed_strat = impute_stratified_median(df, numeric_cols, diag_col)
    imputed_knn = impute_knn_numeric(df, numeric_cols)
    imputed_mice = impute_mice_numeric(df, numeric_cols)
    imputed_missforest = impute_missforest_numeric(df, numeric_cols)

    imputed_by_method = {
        "SIMPLE_MEDIAN": imputed_simple,
        "STRATIFIED_DIAG_MEDIAN": imputed_strat,
        "KNN_NUMERIC": imputed_knn,
        "MICE_NUMERIC": imputed_mice,
        "MISSFOREST_NUMERIC": imputed_missforest,
    }


    final_df = df.copy()
    for row in policy_rows:
        if row["variable_type"] != "numeric":
            continue
        col = row["variable"]
        method = row["chosen_method"]
        if col not in final_df.columns:
            continue
        if method in imputed_by_method:
            final_df[col] = imputed_by_method[method][col]
        else:
            final_df[col] = imputed_simple[col]


    rng = np.random.default_rng(42)
    for row in policy_rows:
        if row["variable_type"] != "numeric":
            continue
        col = row["variable"]
        if not is_core_lab(col):
            continue
        if row["chosen_method"] == "MISSFOREST_NUMERIC":
            apply_jitter_and_clip(final_df, df, col, rng)


    final_df = add_categorical_missing(final_df, categorical_cols)


    final_df = add_numeric_missing_flags(final_df, numeric_cols, df)


    final_df.to_csv(OUTPUT_DIR / "imputed_dataset.csv", index=False)
    try:
        final_df.to_excel(OUTPUT_DIR / "imputed_dataset.xlsx", index=False)
    except Exception:
        pass


    plot_core_lab_histograms(df, final_df, CORE_LABS)


    print("\n=== Methods Summary (thesis-ready) ===")
    print(
        "Categorical variables were imputed by adding an explicit 'Missing' level "
        "to preserve missingness information. Numeric variables were imputed using "
        "a variable-wise policy: non-core variables or those with >40% missingness "
        "were imputed using stratified medians by diagnosis group (or overall median "
        "if diagnosis was unavailable), and core laboratory variables with <=40% "
        "missingness used an ensemble iterative (ExtraTrees) imputer with small "
        "stochastic jitter applied to imputed values and clipping to observed ranges "
        "to avoid distributional spikes. Missingness indicators were added for numeric "
        "variables. A masked-value benchmark (10% masking, 5 repeats, stratified by "
        "diagnosis) was used to inform the policy without using any target variables, "
        "and benchmark scores are reported separately."
    )

    print("\nOutputs saved to:")
    print(f" - {OUTPUT_DIR / 'imputed_dataset.csv'}")
    print(f" - {OUTPUT_DIR / 'imputed_dataset.xlsx'}")
    print(f" - {OUTPUT_DIR / 'imputation_policy.csv'}")
    print(f" - {OUTPUT_DIR / 'imputation_benchmark_summary.csv'}")
    print(f" - {PLOTS_DIR}")


if __name__ == "__main__":
    main()
